# RQ2: Resource Efficiency Analysis

**Research Question:** What is the resource consumption and cost-per-request for each architecture?

## Hypotheses Tested

| ID | Statement | Testable Prediction |
|---|---|---|
| H2a | Monolithic has lowest total resource consumption | monolithic.total_vcpu = 2 < microservices.total_vcpu = 4 |
| H2b | Microservices has lower resource utilization efficiency | microservices.requests_per_cpu_second < monolithic.requests_per_cpu_second |
| H2c | Triton shows higher baseline memory usage | triton.baseline_memory_mb > monolithic.baseline_memory_mb |
| H2d | CPU efficiency converges at high load | variance(efficiency) decreases as concurrent_users increases |

In [ ]:
import sys
from pathlib import Path

# Get project root (works regardless of current working directory)
_notebook_dir = Path().resolve()
if _notebook_dir.name == 'notebooks' and _notebook_dir.parent.name == 'analysis':
    _project_root = _notebook_dir.parent.parent
else:
    _project_root = _notebook_dir

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.utilities.loaders import ResultsLoader

# Load configuration from experiment.yaml
ResultsLoader._load_config_from_yaml()

# Get vCPU allocation from config for reference lines
VCPU_ALLOCATION = ResultsLoader.VCPU_ALLOCATION

# Set publication-quality defaults
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# Output directory for plots
PLOTS_DIR = Path('../plots/rq2')
PLOTS_DIR.mkdir(exist_ok=True, parents=True)


def cohens_d(group1, group2):
    """Calculate Cohen's d effect size between two groups.
    
    Cohen's d = (M1 - M2) / pooled_std
    
    Interpretation:
    - |d| < 0.2: negligible
    - 0.2 <= |d| < 0.5: small
    - 0.5 <= |d| < 0.8: medium
    - |d| >= 0.8: large
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std if pooled_std > 0 else 0


print("vCPU allocation (from experiment.yaml):")
for arch, vcpu in VCPU_ALLOCATION.items():
    print(f"  {arch}: {vcpu} vCPU")

In [ ]:
# Load data with efficiency metrics
loader = ResultsLoader()
df = loader.compute_efficiency_metrics()

print(f"Total runs: {len(df)}")
print(f"\nEfficiency columns added:")
print(f"  - total_vcpu: vCPU allocation per architecture")
print(f"  - throughput_per_vcpu: RPS / vCPU")
print(f"  - cpu_efficiency: actual usage / allocated")
print()
df[['architecture', 'concurrent_users', 'throughput_rps', 'total_vcpu', 'throughput_per_vcpu']].head(10)

In [ ]:
# Data Validation
# Cross-check computed values against expected ranges

# Basic sanity checks
assert df['cpu_avg_percent'].min() >= 0, "Negative CPU usage detected"
assert df['memory_avg_mb'].min() > 0, "Non-positive memory detected"
assert not df['throughput_per_vcpu'].isna().any(), "NaN in efficiency metric"

# Validate vCPU allocation matches experiment.yaml
expected_vcpu = {
    'monolithic': 2,
    'microservices': 4,
    'triton': 4
}
for arch, vcpu in loader.VCPU_ALLOCATION.items():
    assert vcpu == expected_vcpu[arch], f"vCPU mismatch for {arch}: {vcpu} != {expected_vcpu[arch]}"

# Efficiency should be positive and reasonable (0-20 RPS/vCPU typical)
eff_range = df['throughput_per_vcpu']
assert eff_range.min() > 0, "Zero efficiency detected"
assert eff_range.max() < 50, f"Suspiciously high efficiency: {eff_range.max():.1f} RPS/vCPU"

print("Data validation passed")
print(f"  - vCPU allocation matches experiment.yaml")
print(f"  - Efficiency range: {eff_range.min():.2f} - {eff_range.max():.2f} RPS/vCPU")
print(f"  - Total runs: {len(df)}")
print(f"  - Architectures: {df['architecture'].unique().tolist()}")
print(f"  - Load levels: {sorted(df['concurrent_users'].unique().tolist())}")

## 1. Resource Efficiency Bar Chart (H2a, H2b)

Compares requests per second per vCPU across architectures - the "cost of microservices" visualization.

In [ ]:
# Aggregate efficiency by architecture and user count
efficiency_agg = df.groupby(['architecture', 'concurrent_users']).agg({
    'throughput_per_vcpu': ['mean', 'std'],
    'throughput_rps': 'mean',
    'total_vcpu': 'first'
}).reset_index()
efficiency_agg.columns = ['architecture', 'concurrent_users', 'throughput_per_vcpu_mean', 
                          'throughput_per_vcpu_std', 'throughput_rps', 'total_vcpu']

# Select representative load level for bar chart
# Use a moderate load level that shows meaningful efficiency differences
# This is typically where throughput is near peak but before heavy saturation effects
REPRESENTATIVE_LOAD = 25  # Configurable: adjust based on experiment's throughput curve

peak_load = efficiency_agg[efficiency_agg['concurrent_users'] == REPRESENTATIVE_LOAD]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(loader.ARCH_DISPLAY_NAMES))
width = 0.6

bars = []
for i, arch in enumerate(loader.ARCH_DISPLAY_NAMES.keys()):
    arch_data = peak_load[peak_load['architecture'] == arch]
    if len(arch_data) > 0:
        bar = ax.bar(i, arch_data['throughput_per_vcpu_mean'].values[0],
                     width, yerr=arch_data['throughput_per_vcpu_std'].values[0],
                     color=loader.ARCH_COLORS[arch], capsize=5,
                     label=loader.ARCH_DISPLAY_NAMES[arch])
        bars.append(bar)
        
        # Add value annotation
        val = arch_data['throughput_per_vcpu_mean'].values[0]
        ax.annotate(f'{val:.2f}', xy=(i, val), ha='center', va='bottom', fontsize=11)

ax.set_xlabel('Architecture')
ax.set_ylabel('Throughput per vCPU (RPS/vCPU)')
ax.set_title(f'Resource Efficiency: Requests per Second per vCPU (at {REPRESENTATIVE_LOAD} users)')
ax.set_xticks(x)
ax.set_xticklabels([loader.ARCH_DISPLAY_NAMES[a] for a in loader.ARCH_DISPLAY_NAMES.keys()])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq2_efficiency_bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq2_efficiency_bar_chart.png")

## 2. Efficiency Trend Across Load Levels

Shows how efficiency changes with load (H2d: convergence at high load).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for arch in loader.ARCH_DISPLAY_NAMES.keys():
    arch_data = efficiency_agg[efficiency_agg['architecture'] == arch]
    ax.errorbar(
        arch_data['concurrent_users'],
        arch_data['throughput_per_vcpu_mean'],
        yerr=arch_data['throughput_per_vcpu_std'],
        label=f"{loader.ARCH_DISPLAY_NAMES[arch]} ({int(arch_data['total_vcpu'].iloc[0])} vCPU)",
        color=loader.ARCH_COLORS[arch],
        marker='o',
        capsize=3,
        linewidth=2,
        markersize=6
    )

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('Throughput per vCPU (RPS/vCPU)')
ax.set_title('Resource Efficiency Across Load Levels')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xticks(loader.USER_LEVELS)

# Highlight high concurrency region for H2d
ax.axvspan(50, 100, alpha=0.1, color='yellow')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq2_efficiency_trend.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq2_efficiency_trend.png")

## 3. Network Bandwidth Comparison (Triton overhead)

Shows the dramatic difference in network I/O between Triton and custom implementations.

In [ ]:
# Aggregate network metrics
network_agg = df.groupby(['architecture', 'concurrent_users']).agg({
    'network_rx_bytes_per_sec': ['mean', 'std'],
    'network_tx_bytes_per_sec': ['mean', 'std']
}).reset_index()
network_agg.columns = ['architecture', 'concurrent_users', 'rx_mean', 'rx_std', 'tx_mean', 'tx_std']

# Convert to MB/s for readability
network_agg['rx_mean_mbps'] = network_agg['rx_mean'] / (1024 * 1024)
network_agg['tx_mean_mbps'] = network_agg['tx_mean'] / (1024 * 1024)
network_agg['rx_std_mbps'] = network_agg['rx_std'] / (1024 * 1024)
network_agg['tx_std_mbps'] = network_agg['tx_std'] / (1024 * 1024)

In [ ]:
# Bar chart at representative load (25 users)
peak_network = network_agg[network_agg['concurrent_users'] == 25]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(loader.ARCH_DISPLAY_NAMES))
width = 0.35

for i, arch in enumerate(loader.ARCH_DISPLAY_NAMES.keys()):
    arch_data = peak_network[peak_network['architecture'] == arch]
    if len(arch_data) > 0:
        rx = arch_data['rx_mean_mbps'].values[0]
        tx = arch_data['tx_mean_mbps'].values[0]
        rx_err = arch_data['rx_std_mbps'].values[0]
        tx_err = arch_data['tx_std_mbps'].values[0]
        
        ax.bar(i - width/2, rx, width, yerr=rx_err, color=loader.ARCH_COLORS[arch], 
               alpha=0.8, capsize=3, label='RX' if i == 0 else '')
        ax.bar(i + width/2, tx, width, yerr=tx_err, color=loader.ARCH_COLORS[arch], 
               alpha=0.4, capsize=3, label='TX' if i == 0 else '', hatch='//')

ax.set_xlabel('Architecture')
ax.set_ylabel('Network I/O (MB/s)')
ax.set_title('Network Bandwidth by Architecture (at 25 users)')
ax.set_xticks(x)
ax.set_xticklabels([loader.ARCH_DISPLAY_NAMES[a] for a in loader.ARCH_DISPLAY_NAMES.keys()])
ax.legend(['Receive (RX)', 'Transmit (TX)'])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq2_network_bandwidth_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq2_network_bandwidth_bar.png")

In [ ]:
# Network RX trend across load levels
fig, ax = plt.subplots(figsize=(10, 6))

for arch in loader.ARCH_DISPLAY_NAMES.keys():
    arch_data = network_agg[network_agg['architecture'] == arch]
    ax.errorbar(
        arch_data['concurrent_users'],
        arch_data['rx_mean_mbps'],
        yerr=arch_data['rx_std_mbps'],
        label=loader.ARCH_DISPLAY_NAMES[arch],
        color=loader.ARCH_COLORS[arch],
        marker='o',
        capsize=3,
        linewidth=2,
        markersize=6
    )

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('Network Receive (MB/s)')
ax.set_title('Network I/O Scaling by Architecture')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xticks(loader.USER_LEVELS)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq2_network_trend.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq2_network_trend.png")

## 4. CPU Utilization Analysis

In [ ]:
# CPU utilization trend
cpu_agg = df.groupby(['architecture', 'concurrent_users']).agg({
    'cpu_avg_percent': ['mean', 'std'],
    'cpu_max_percent': 'mean',
    'total_vcpu': 'first'
}).reset_index()
cpu_agg.columns = ['architecture', 'concurrent_users', 'cpu_avg_mean', 'cpu_avg_std', 'cpu_max', 'total_vcpu']

# Normalize to per-vCPU percentage (100% = 1 vCPU fully utilized)
cpu_agg['cpu_per_vcpu'] = cpu_agg['cpu_avg_mean'] / cpu_agg['total_vcpu']

fig, ax = plt.subplots(figsize=(10, 6))

for arch in loader.ARCH_DISPLAY_NAMES.keys():
    arch_data = cpu_agg[cpu_agg['architecture'] == arch]
    ax.errorbar(
        arch_data['concurrent_users'],
        arch_data['cpu_avg_mean'],
        yerr=arch_data['cpu_avg_std'],
        label=f"{loader.ARCH_DISPLAY_NAMES[arch]} ({int(arch_data['total_vcpu'].iloc[0])} vCPU)",
        color=loader.ARCH_COLORS[arch],
        marker='o',
        capsize=3,
        linewidth=2,
        markersize=6
    )

# Add allocation reference lines - computed from VCPU_ALLOCATION (100% = 1 vCPU)
mono_allocation_pct = VCPU_ALLOCATION['monolithic'] * 100
micro_allocation_pct = VCPU_ALLOCATION['microservices'] * 100

ax.axhline(y=mono_allocation_pct, color='#2ecc71', linestyle='--', alpha=0.5, 
           label=f'Mono allocation ({VCPU_ALLOCATION["monolithic"]} vCPU)')
ax.axhline(y=micro_allocation_pct, color='#3498db', linestyle='--', alpha=0.5, 
           label=f'Micro/Triton allocation ({VCPU_ALLOCATION["microservices"]} vCPU)')

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('CPU Utilization (%)')
ax.set_title('CPU Utilization by Architecture')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xticks(loader.USER_LEVELS)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq2_cpu_utilization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq2_cpu_utilization.png")

## 5. Memory Usage Analysis (H2c)

In [ ]:
# Memory usage at lowest load (baseline indicator)
memory_agg = df.groupby(['architecture', 'concurrent_users']).agg({
    'memory_avg_mb': ['mean', 'std'],
    'memory_max_mb': 'mean'
}).reset_index()
memory_agg.columns = ['architecture', 'concurrent_users', 'memory_avg_mean', 'memory_avg_std', 'memory_max']

fig, ax = plt.subplots(figsize=(10, 6))

for arch in loader.ARCH_DISPLAY_NAMES.keys():
    arch_data = memory_agg[memory_agg['architecture'] == arch]
    ax.errorbar(
        arch_data['concurrent_users'],
        arch_data['memory_avg_mean'],
        yerr=arch_data['memory_avg_std'],
        label=loader.ARCH_DISPLAY_NAMES[arch],
        color=loader.ARCH_COLORS[arch],
        marker='o',
        capsize=3,
        linewidth=2,
        markersize=6
    )

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('Memory Usage (MB)')
ax.set_title('Memory Usage by Architecture')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xticks(loader.USER_LEVELS)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq2_memory_usage.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq2_memory_usage.png")

## 6. Hypothesis Validation

In [ ]:
# H2a: Monolithic lowest total resource consumption
# Note: This is a design constraint, not a statistical comparison - effect size is N/A
print("H2a: Total Resource Allocation")
print("="*50)
for arch, vcpu in loader.VCPU_ALLOCATION.items():
    print(f"{arch}: {vcpu} vCPU")

h2a_supported = loader.VCPU_ALLOCATION['monolithic'] < loader.VCPU_ALLOCATION['microservices']
print(f"\nH2a Supported (mono < micro): {h2a_supported}")
print("\nNote: H2a is a design constraint (2 vs 4 vCPU), not a statistical comparison.")
print("Effect size: N/A (design constraint)")

In [ ]:
# H2b: Microservices lower efficiency than monolithic
print("\nH2b: Resource Efficiency Comparison")
print("="*50)

efficiency_by_arch = df.groupby('architecture')['throughput_per_vcpu'].mean()
print("Average throughput per vCPU (RPS/vCPU):")
print(efficiency_by_arch.sort_values(ascending=False))

mono_eff = efficiency_by_arch['monolithic']
micro_eff = efficiency_by_arch['microservices']

# Calculate Cohen's d for efficiency comparison
mono_eff_data = df[df['architecture'] == 'monolithic']['throughput_per_vcpu']
micro_eff_data = df[df['architecture'] == 'microservices']['throughput_per_vcpu']
h2b_d = cohens_d(mono_eff_data, micro_eff_data)

h2b_supported = micro_eff < mono_eff
print(f"\nH2b Supported (micro_eff < mono_eff): {h2b_supported}")
print(f"Efficiency gap: {((mono_eff - micro_eff) / mono_eff * 100):.1f}%")
print(f"\nCohen's d effect size: {h2b_d:.2f}")
print(f"Interpretation: {'large' if abs(h2b_d) >= 0.8 else 'medium' if abs(h2b_d) >= 0.5 else 'small' if abs(h2b_d) >= 0.2 else 'negligible'}")

In [ ]:
# H2c: Triton higher baseline memory
print("\nH2c: Baseline Memory Usage (at 1 user)")
print("="*50)

baseline_memory = df[df['concurrent_users'] == 1].groupby('architecture')['memory_avg_mb'].mean()
print(baseline_memory.sort_values(ascending=False))

triton_mem = baseline_memory.get('triton', 0)
mono_mem = baseline_memory.get('monolithic', 0)

# Calculate Cohen's d for baseline memory comparison
triton_mem_data = df[(df['architecture'] == 'triton') & (df['concurrent_users'] == 1)]['memory_avg_mb']
mono_mem_data = df[(df['architecture'] == 'monolithic') & (df['concurrent_users'] == 1)]['memory_avg_mb']
h2c_d = cohens_d(triton_mem_data, mono_mem_data)

h2c_supported = triton_mem > mono_mem
print(f"\nH2c Supported (triton > monolithic): {h2c_supported}")
print(f"Triton overhead: {triton_mem - mono_mem:.1f} MB ({((triton_mem - mono_mem) / mono_mem * 100):.1f}%)")
print(f"\nCohen's d effect size: {h2c_d:.2f}")
print(f"Interpretation: {'large' if abs(h2c_d) >= 0.8 else 'medium' if abs(h2c_d) >= 0.5 else 'small' if abs(h2c_d) >= 0.2 else 'negligible'}")

In [ ]:
# H2d: Efficiency convergence at high load
print("\nH2d: Efficiency Variance at Different Load Levels")
print("="*50)

# Calculate variance of efficiency across architectures at each load level
variance_by_load = df.groupby('concurrent_users').apply(
    lambda x: x.groupby('architecture')['throughput_per_vcpu'].mean().var()
).reset_index(name='efficiency_variance')

print(variance_by_load)

# Check if variance decreases at high load (>=50 users)
low_load_var = variance_by_load[variance_by_load['concurrent_users'] < 50]['efficiency_variance'].mean()
high_load_var = variance_by_load[variance_by_load['concurrent_users'] >= 50]['efficiency_variance'].mean()

# Calculate effect size on variance comparison
low_load_variances = variance_by_load[variance_by_load['concurrent_users'] < 50]['efficiency_variance']
high_load_variances = variance_by_load[variance_by_load['concurrent_users'] >= 50]['efficiency_variance']
h2d_d = cohens_d(low_load_variances, high_load_variances)

h2d_supported = high_load_var < low_load_var
print(f"\nLow load (<50) variance: {low_load_var:.4f}")
print(f"High load (>=50) variance: {high_load_var:.4f}")
print(f"H2d Supported (variance decreases at high load): {h2d_supported}")
print(f"\nCohen's d effect size: {h2d_d:.2f}")
print(f"Interpretation: {'large' if abs(h2d_d) >= 0.8 else 'medium' if abs(h2d_d) >= 0.5 else 'small' if abs(h2d_d) >= 0.2 else 'negligible'}")

## 7. Summary Statistics Table

Table showing mean and standard deviation for key metrics at representative load levels (1, 10, 50, 100 users).

In [ ]:
# Generate summary table with SD columns for thesis
# Use 4 representative load levels per CONTEXT.md
REPRESENTATIVE_LEVELS = [1, 10, 50, 100]

df_representative = df[df['concurrent_users'].isin(REPRESENTATIVE_LEVELS)]

summary_stats = df_representative.groupby(['architecture', 'concurrent_users']).agg({
    'cpu_avg_percent': ['mean', 'std'],
    'memory_avg_mb': ['mean', 'std'],
    'throughput_per_vcpu': ['mean', 'std'],
    'network_rx_bytes_per_sec': ['mean', 'std']
})

# Flatten column names and format
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns]
summary_stats = summary_stats.reset_index()

# Convert network to MB/s
summary_stats['network_rx_bytes_per_sec_mean'] = summary_stats['network_rx_bytes_per_sec_mean'] / (1024 * 1024)
summary_stats['network_rx_bytes_per_sec_std'] = summary_stats['network_rx_bytes_per_sec_std'] / (1024 * 1024)

# Create formatted table with SD columns
summary_table = pd.DataFrame({
    'Architecture': summary_stats['architecture'].apply(lambda x: loader.ARCH_DISPLAY_NAMES[x]),
    'Users': summary_stats['concurrent_users'],
    'CPU Avg (%)': summary_stats['cpu_avg_percent_mean'].round(2),
    'CPU SD': summary_stats['cpu_avg_percent_std'].round(2),
    'Memory (MB)': summary_stats['memory_avg_mb_mean'].round(1),
    'Memory SD': summary_stats['memory_avg_mb_std'].round(1),
    'RPS/vCPU': summary_stats['throughput_per_vcpu_mean'].round(2),
    'RPS/vCPU SD': summary_stats['throughput_per_vcpu_std'].round(2),
    'Network RX (MB/s)': summary_stats['network_rx_bytes_per_sec_mean'].round(2),
    'Network RX SD': summary_stats['network_rx_bytes_per_sec_std'].round(2)
})

print("\nRQ2 Summary Table (Mean with Standard Deviation)")
print("="*100)
print(summary_table.to_string(index=False))

# Export to CSV for thesis
summary_table.to_csv(PLOTS_DIR / 'rq2_summary_table.csv', index=False)
print("\nSaved: rq2_summary_table.csv")

## 8. Hypothesis Results Summary

### Effect Size Interpretation (Cohen's d)

Cohen's d provides a standardized measure of effect size, independent of sample size:

| |d| Range | Interpretation |
|-----------|----------------|
| < 0.2 | Negligible |
| 0.2 - 0.5 | Small |
| 0.5 - 0.8 | Medium |
| >= 0.8 | Large |

**Note on statistical limitations:** With n=3 runs per configuration, effect sizes provide practical significance guidance but should be interpreted with caution. The variance estimates may be unstable with small samples.

In [ ]:
def interp_d(d):
    """Interpret Cohen's d magnitude."""
    if d is None or d == 'N/A':
        return 'N/A'
    d = abs(d)
    if d < 0.2:
        return 'negligible'
    elif d < 0.5:
        return 'small'
    elif d < 0.8:
        return 'medium'
    else:
        return 'large'

# Build comprehensive hypothesis results table
hypothesis_results = pd.DataFrame([
    {
        'Hypothesis': 'H2a',
        'Statement': 'Monolithic lowest resource allocation',
        'Predicted': 'mono (2 vCPU) < micro (4 vCPU)',
        'Observed': f"mono={loader.VCPU_ALLOCATION['monolithic']}, micro={loader.VCPU_ALLOCATION['microservices']}",
        'Supported': h2a_supported,
        'Effect_Size': 'N/A (design)',
        'Interpretation': 'N/A'
    },
    {
        'Hypothesis': 'H2b',
        'Statement': 'Microservices lower efficiency',
        'Predicted': 'micro_eff < mono_eff',
        'Observed': f'{micro_eff:.2f} vs {mono_eff:.2f} RPS/vCPU',
        'Supported': h2b_supported,
        'Effect_Size': f'{h2b_d:.2f}',
        'Interpretation': interp_d(h2b_d)
    },
    {
        'Hypothesis': 'H2c',
        'Statement': 'Triton higher baseline memory',
        'Predicted': 'triton > monolithic (at 1 user)',
        'Observed': f'{triton_mem:.1f} vs {mono_mem:.1f} MB',
        'Supported': h2c_supported,
        'Effect_Size': f'{h2c_d:.2f}',
        'Interpretation': interp_d(h2c_d)
    },
    {
        'Hypothesis': 'H2d',
        'Statement': 'Efficiency converges at high load',
        'Predicted': 'variance(eff) at high < low load',
        'Observed': f'var(high)={high_load_var:.4f} vs var(low)={low_load_var:.4f}',
        'Supported': h2d_supported,
        'Effect_Size': f'{h2d_d:.2f}',
        'Interpretation': interp_d(h2d_d)
    },
])

print("\nRQ2 Hypothesis Results")
print("="*120)
print(hypothesis_results.to_string(index=False))

hypothesis_results.to_csv(PLOTS_DIR / 'rq2_hypothesis_results.csv', index=False)
print("\nSaved: rq2_hypothesis_results.csv")